In [6]:
%pwd

'/Users/johnlee/code/gcp-work/platform/mlflow'

In [8]:
from google_auth_oauthlib.flow import InstalledAppFlow
import requests
import json
import time
import os
import glob
import sys
import os, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from ignite.engine import Engine, Events
from ignite.metrics import Accuracy, Loss
from ignite.handlers.mlflow_logger import (
    MLflowLogger, OutputHandler, OptimizerParamsHandler, global_step_from_engine
)

import mlflow

In [9]:
# Disable OAuth scope validation (same as diagnostic script)
os.environ['OAUTHLIB_RELAX_TOKEN_SCOPE'] = '1'

# 1) Find client secret file using glob (same as diagnostic script)
client_secret_patterns = glob.glob("client_secret*.json", recursive=True)
if not client_secret_patterns:
    raise FileNotFoundError("No client_secret*.json file found. Please ensure it exists in the current directory or subdirectories.")

client_secret_file = client_secret_patterns[0]
print(f"Using client secret: {client_secret_file}")

# 2) Start the installed-app flow (use your desktop client secrets JSON)
#    IMPORTANT: Use same scopes as diagnostic script that worked!
flow = InstalledAppFlow.from_client_secrets_file(
    client_secret_file,
    scopes=["openid", "email", "profile"]  # Fixed: added "email", "profile" scopes
)

creds = flow.run_local_server(port=0)  # opens browser and returns creds
id_token = creds.id_token
print(f"✅ Got id_token (len): {len(id_token)}")

# 3) MLflow tracking base URL (oauth2-proxy fronted)
MLFLOW_BASE = "https://mlflow.cervical-screening.pythonaisolutions.com"
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_BASE
os.environ["MLFLOW_TRACKING_TOKEN"] = id_token  # Set token for MLflow client

Using client secret: client_secret_1098946209440-c5crpupjjv3p179ok6n5snc37s0fvd16.apps.googleusercontent.com.json
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1098946209440-c5crpupjjv3p179ok6n5snc37s0fvd16.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A61677%2F&scope=openid+email+profile&state=d1o3X38i2FQ2fGX6cSJChiIwZCULie&access_type=offline
✅ Got id_token (len): 1190


In [10]:

mlflow.set_experiment(
  experiment_name="ignite-notebook-demo",
)

# Dummy data/model just to demonstrate logging
x = torch.randn(1024, 20)
y = torch.randint(0, 2, (1024,))
train_loader = DataLoader(TensorDataset(x, y), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(x, y), batch_size=64)

model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 2))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train_step(engine, batch):
    model.train()
    xb, yb = batch
    optimizer.zero_grad()
    logits = model(xb)
    loss = criterion(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item()

def eval_step(engine, batch):
    model.eval()
    with torch.no_grad():
        xb, yb = batch
        logits = model(xb)
        return logits, yb

trainer  = Engine(train_step)
evaluator = Engine(eval_step)

Accuracy(output_transform=lambda out: (out[0].argmax(dim=1), out[1])).attach(evaluator, "accuracy")
Loss(criterion, output_transform=lambda out: (out[0], out[1])).attach(evaluator, "loss")

# MLflow logger (picks up MLFLOW_TRACKING_URI + MLFLOW_TRACKING_TOKEN from the auth cell)
mlf = MLflowLogger()

# Optional: params/tags for easier comparison in the UI
mlf.log_params({
    "model": type(model).__name__,
    "optimizer": type(optimizer).__name__,
    "lr": optimizer.param_groups[0]["lr"],
    "platform": "notebook",
})

# Log training loss every iteration
mlf.attach(
    trainer,
    log_handler=OutputHandler(tag="train", output_transform=lambda loss: {"loss": loss}),
    event_name=Events.ITERATION_COMPLETED,
)

# Log validation metrics each epoch (aligned to trainer's global step)
mlf.attach(
    evaluator,
    log_handler=OutputHandler(
        tag="val",
        metric_names=["accuracy", "loss"],
        global_step_transform=global_step_from_engine(trainer)
    ),
    event_name=Events.EPOCH_COMPLETED,
)

# Log optimizer LR over time
mlf.attach(
    trainer,
    log_handler=OptimizerParamsHandler(optimizer),
    event_name=Events.ITERATION_STARTED,
)

# Save & log a checkpoint each epoch (goes to your MLflow artifact store)
@trainer.on(Events.EPOCH_COMPLETED)
def _checkpoint(_):
    torch.save(model.state_dict(), "weights.pt")
    mlf.log_artifact("weights.pt")

trainer.run(train_loader, max_epochs=3)
evaluator.run(val_loader)
mlf.close()

print("✅ Logged run to:", os.environ.get("MLFLOW_TRACKING_URI"))

2025/10/29 20:38:44 INFO mlflow.tracking.fluent: Experiment with name 'ignite-notebook-demo' does not exist. Creating a new experiment.


🏃 View run auspicious-deer-565 at: https://mlflow.cervical-screening.pythonaisolutions.com/#/experiments/2/runs/4f006fea71564b4fadf639cfcce0c494
🧪 View experiment at: https://mlflow.cervical-screening.pythonaisolutions.com/#/experiments/2
✅ Logged run to: https://mlflow.cervical-screening.pythonaisolutions.com
